In [20]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point


# Region boundaries and random weather locations

In NVE Atlas, we retrieve Budområder and NVE_Region from NVE's REST services (????) and save them as shapefiles.

The assignment's precipitation regions are grouped as East (Øst) = RS + RØ, West (Vest) = RV, and North (Nord) = RM + RN. These are groups of NVE regions (!!!)

----------------------------------

## Retreive the NVE regions, then rainfall areas

Read the shapefil using geopandas and using the encoding as "utf-8" to handle Norwegian characters correctly.

In [21]:
regions = gpd.read_file("../data/shapefiles/NVE_region.shp", encoding = 'utf-8')
# . means current directory, .. means parent directory

print("Coordinate Reference System:", regions.crs)  # Coordinate Reference System
print("Column names:", regions.columns.tolist())  # List of column names

regions

Coordinate Reference System: EPSG:25833
Column names: ['OBJECTID', 'REGION', 'REGIONNAVN', 'POSTNUMMER', 'POSTSTED', 'BESØKSADR', 'ADRESSE', 'geometry']


,OBJECTID,REGION,REGIONNAVN,POSTNUMMER,POSTSTED,BESØKSADR,ADRESSE,geometry
0,1.0,RV,Region Vest,6801,Førde,Naustdalsveien 1B,Postboks 53,"POLYGON ((193105.76 6929797.839, 191705.75 692..."
1,2.0,RM,Region Midt-Norge,7030,Trondheim,"Abelsgate 9, Trondheim",Abelsgate 9,"POLYGON ((391250 7388620, 392270 7386790, 3937..."
2,3.0,RN,Region Nord,8514,Narvik,Kongensgate 52-54,Kongensgate 52-54,"POLYGON ((1109096.413 7844647.502, 1105115.343..."
3,4.0,RØ,Region Øst,2307,Hamar,Vangsvegen 73,Postboks 4223,"POLYGON ((253748.22 6678130.91, 253645.58 6678..."
4,5.0,RS,Region Sør,3103,Tønsberg,Anton Jenssens gate 7,Postboks 2124,"POLYGON ((253748.22 6678130.91, 253917.63 6677..."


There are five administrative NVE regions in total, which will be used to combined into three rainfall areas:
- East (Øst) = RS + RØ
- West (Vest) = RV
- North (Nord) = RM + RN

In [22]:
# Map NVE administrative regions to precipitation regions
region_mapping = {
    "RS": "Øst",
    "RØ": "Øst",
    "RV": "Vest",
    "RM": "Nord",
    "RN": "Nord"
}

# Assign precipitation region names
precipitation_regions = regions.copy()
precipitation_regions["region"] = precipitation_regions["REGION"].map(region_mapping)

# Merge polygons belonging to the same precipitation region
precipitation_regions = precipitation_regions.dissolve(
    by="region",
    as_index=False
)

precipitation_regions[["region"]]

,region
0,Nord
1,Vest
2,Øst


## Retrieve the power zones (strømområdene)

In [23]:
electricity_areas = gpd.read_file(
    "../data/shapefiles/Budområder.shp"
)

print("CRS:", electricity_areas.crs)
print("Columns:", electricity_areas.columns.tolist())
electricity_areas

CRS: EPSG:25833
Columns: ['objectid', 'budomr', 'globalid', 'st_area(sh', 'st_perimet', 'geometry']


,objectid,budomr,globalid,st_area(sh,st_perimet,geometry
0,1.0,NO 1,NaN,6.052902e+10,1.657761e+06,"POLYGON ((337473.63 6970542.97, 337711.9 69703..."
1,2.0,NO 2,NaN,6.875969e+10,1.256473e+06,"POLYGON ((-9833.91 6726499.86, -9836 6726404.0..."
2,3.0,NO 3,NaN,1.042199e+11,1.773323e+06,"POLYGON ((308588.7 7261519.43, 313576.63 72590..."
3,4.0,NO 4,NaN,2.032500e+11,3.385499e+06,"POLYGON ((973920.93 7962739.19, 974922.77 7962..."
4,5.0,NO 5,NaN,3.334870e+10,1.106762e+06,"POLYGON ((101466.43 6881318.54, 101525.43 6881..."


There are five regions for power zones

## Generate random points for each region 

Combine power and rainfall areas to use for weather data download

In [24]:
# Standardize electricity region names
electricity_areas["region"] = (
    electricity_areas["budomr"]
    .str.replace(" ", "", regex=False)
)

# Keep only region names and geometries
electricity = electricity_areas[["region", "geometry"]]
precipitation = precipitation_regions[["region", "geometry"]]

# Ensure both datasets use the same coordinate system
precipitation = precipitation.to_crs(electricity.crs)

# Combine all eight regions
all_regions = gpd.GeoDataFrame(
    pd.concat([electricity, precipitation], ignore_index=True),
    geometry="geometry",
    crs=electricity.crs
)

print(all_regions["region"].tolist())

['NO1', 'NO2', 'NO3', 'NO4', 'NO5', 'Nord', 'Vest', 'Øst']


In [27]:
import geopandas as gpd
import pandas as pd

def generate_weather_locations(regions, n_points=1000, seed=42):
    """
    Generate random weather locations for each geographic region.
    """

    # Use an equal-area projection for spatial sampling
    regions = regions.to_crs(epsg=3035).copy()

    # Sample random points within each region
    sampled = regions[["region", "geometry"]].copy()
    sampled["geometry"] = regions.geometry.sample_points(
        size=n_points,
        rng=seed
    )

    # Convert multipoints into individual rows
    sampled = sampled.explode(
        index_parts=False
    ).reset_index(drop=True)

    # Assign a unique point ID within each region
    sampled["point_id"] = (
        sampled.groupby("region").cumcount() + 1
    )

    # Convert coordinates to WGS84 for Open-Meteo
    sampled = sampled.to_crs(epsg=4326)

    sampled["longitude"] = sampled.geometry.x
    sampled["latitude"] = sampled.geometry.y

    return sampled


weather_locations = generate_weather_locations(
    all_regions,
    n_points=1000,
    seed=42
)

print(weather_locations.groupby("region").size())

region
NO1     1000
NO2     1000
NO3     1000
NO4     1000
NO5     1000
Nord    1000
Vest    1000
Øst     1000
dtype: int64


Save the coordinates in csv file

## Get historical weather data from Open-Meteo (ERA5)

In [ ]:
import requests
import pandas as pd

def get_era5_weather(latitude, longitude, start_date, end_date):
    """
    Download hourly ERA5 reanalysis weather data
    for a single geographic location.
    """

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation",
        "models": "era5",
        "timezone": "UTC"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    # The response is expected to be in JSON format, and we extract the "hourly" data
    data = response.json()["hourly"]

    df = pd.DataFrame(data)
    # Convert the "time" column to datetime format with UTC timezone
    df["time"] = pd.to_datetime(df["time"], utc=True)

    return df

## Get ERA5-data based on region code

In [ ]:
def get_region_weather(
    region,
    start_date,
    end_date,
    points_df=random_points,
    n_locations=1
):
    """
    Download hourly ERA5 weather data for locations
    sampled within a specified NVE region.

    """

    # Select locations belonging to the region
    region_points = points_df[
        points_df["REGION"] == region
    ]

    if region_points.empty:
        raise ValueError(f"Unknown region: {region}")

    if not 1 <= n_locations <= len(region_points):
        raise ValueError("Invalid number of locations")

    results = []

    for _, point in region_points.head(n_locations).iterrows():

        weather = get_era5_weather(
            latitude=point["latitude"],
            longitude=point["longitude"],
            start_date=start_date,
            end_date=end_date
        )

        weather["REGION"] = region
        weather["latitude"] = point["latitude"]
        weather["longitude"] = point["longitude"]

        results.append(weather)

    return pd.concat(results, ignore_index=True)

In [ ]:
weather_rm = get_region_weather(
    region="RM", # Test with one of the regions (e.g., "RM" for Region Midt-Norge)
    start_date="2024-01-01",
    end_date="2024-12-31",
    n_locations=1
)

weather_rm

,time,temperature_2m,precipitation,REGION,latitude,longitude
0,2024-01-01 00:00:00+00:00,-3.9,0.0,RM,64.020179,10.35306
1,2024-01-01 01:00:00+00:00,-4.0,0.0,RM,64.020179,10.35306
2,2024-01-01 02:00:00+00:00,-3.9,0.0,RM,64.020179,10.35306
3,2024-01-01 03:00:00+00:00,-3.7,0.0,RM,64.020179,10.35306
4,2024-01-01 04:00:00+00:00,-3.6,0.0,RM,64.020179,10.35306
...,...,...,...,...,...,...
8779,2024-12-31 19:00:00+00:00,-4.1,0.0,RM,64.020179,10.35306
8780,2024-12-31 20:00:00+00:00,-5.3,0.0,RM,64.020179,10.35306
8781,2024-12-31 21:00:00+00:00,-5.8,0.0,RM,64.020179,10.35306
8782,2024-12-31 22:00:00+00:00,-6.4,0.0,RM,64.020179,10.35306
